# 数据获取说明与代码（成员 1 的配套文件）

本项目**全部原始数据**是怎么获取的，都在这一个文件里讲清楚：

- **主数据源：CSMAR 国泰安官方 API**（`csmarapi`，题目 Primary method）：
  三张财务报表 `FS_Combas / FS_Comins / FS_Comscfd`、公告日期 `IAR_Rept`、财务指标库 `FI_T1/T4/T5/T6/T8`、
  公司信息 `TRD_Co`、官方行情 `TRD_Dalyr`（含 `Dretwd` 日收益率 / `Adjprcwd` 后复权价 / `Dsmvtll` 总市值·千元）
- **补充数据源：东方财富公开接口**（`requests` + JSON，Class03 Part A 的 Web API 用法；题目 Alternative method 允许）：
  日线三套复权口径（bfq / qfq / hfq）、沪深300、上证综指、分红送转明细；股本变动用 akshare（东方财富底层）
- **为何需要补充**：CSMAR 官方接口有**单日下载配额限制**，配额不足时段用东财补充；
  补充数据已与 CSMAR 官方口径**逐日交叉验证**（`Dretwd` 相关系数 0.99999989，见 `output/compare_csmar_daily.csv`）
- 原始数据已下载保存在本文件夹 `数据/`，本文件**默认不执行**（`RUN_DOWNLOAD = False`），仅展示获取方法；
  完整取数脚本见 `B_我的重做版本/code/step01~step15`（含限速重试、断点续传）。

In [1]:
from __future__ import annotations
import io, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from types import SimpleNamespace

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

BASE = os.getcwd()
RAW = os.path.join(BASE, '数据')
OUT = os.path.join(BASE, '输出')
os.makedirs(OUT, exist_ok=True)
STOCK_NAME = '牧原股份'
STOCK_CODE = '002714.SZ'
TRADING_DAYS = 243


In [2]:
# ============================================================
# 全部数据获取代码（演示）——主数据源 CSMAR，东财仅补充未覆盖字段/时段
# 本 cell 默认不执行（RUN_DOWNLOAD = False），原始数据已在 数据/ 目录。
# CSMAR 官方 API 有单日下载配额限制：重跑请选配额充足时段。
# ============================================================
RUN_DOWNLOAD = False

if RUN_DOWNLOAD:
    import os, pandas as pd, requests

    # ---------- 1) CSMAR 官方 API（主数据源：财务报表/公告日/比率库/公司信息/官方行情） ----------
    from csmarapi.CsmarService import CsmarService
    cred = [x.strip() for x in open(r'../../0_题目与数据接入/CSMAR账号密码.txt', encoding='utf-8') if x.strip()]
    csmar = CsmarService()
    csmar.login(cred[0], cred[1], '0')          # 第三个参数 '0'：返回中文库名（脚本只读，不打印密码）

    plan = {'FS_Combas': 'fs_combas.csv', 'FS_Comins': 'fs_comins.csv', 'FS_Comscfd': 'fs_comscfd.csv',
            'IAR_Rept': 'iar_rept.csv', 'TRD_Co': 'co_info.csv',
            'FI_T1': 'fi_t1_solvency.csv', 'FI_T4': 'fi_t4_efficiency.csv',
            'FI_T5': 'fi_t5_profitability.csv', 'FI_T6': 'fi_t6_cashflow.csv', 'FI_T8': 'fi_t8_growth.csv'}
    for table, fname in plan.items():
        fields = [f['field'] for f in csmar.getListFields(table)]   # 先列字段，再按需取数
        df = csmar.query_df(fields, "Stkcd='002714'", table, '2011-01-01', '2026-06-30')
        df.to_csv(os.path.join(RAW, fname), index=False, encoding='utf-8-sig')

    # 官方行情 TRD_Dalyr：Clsprc 收盘 / Dretwd 日收益率(含红利再投资) / Adjprcwd 后复权可比价 / Dsmvtll 总市值(千元)
    dalyr = csmar.query_df(
        ['Stkcd', 'Trddt', 'Opnprc', 'Hiprc', 'Loprc', 'Clsprc', 'Dnshrtrd', 'Dnvaltrd',
         'Dsmvosd', 'Dsmvtll', 'Dretwd', 'Dretnd', 'Adjprcwd', 'Adjprcnd'],
        "Stkcd='002714'", 'TRD_Dalyr', '2014-01-01', '2026-10-07')
    dalyr.to_csv(os.path.join(RAW, 'dalyr_price.csv'), index=False, encoding='utf-8-sig')

    # ---------- 2) 东方财富公开接口（补充 CSMAR 未覆盖的字段与时段；requests + JSON） ----------
    URL = 'https://push2his.eastmoney.com/api/qt/stock/kline/get'
    HEAD = {'User-Agent': 'Mozilla/5.0'}
    COLS = ['日期', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']
    def kline(secid, fqt, out):
        r = requests.get(URL, params={'secid': secid, 'fields1': 'f1,f2,f3,f4,f5,f6',
                                      'fields2': 'f51,f52,f53,f54,f55,f56,f57,f58,f59,f60,f61',
                                      'klt': '101', 'fqt': fqt, 'beg': '20140101', 'end': '20261007'},
                         headers=HEAD, timeout=30)
        df = pd.DataFrame([k.split(',') for k in r.json()['data']['klines']], columns=COLS)
        df.to_csv(os.path.join(RAW, out), index=False, encoding='utf-8-sig')
    for fqt, nm in [('0', 'bfq'), ('1', 'qfq'), ('2', 'hfq')]:
        kline('0.002714', fqt, 'em_002714_%s.csv' % nm)      # 牧原日线三种复权
    kline('1.000300', '1', 'em_index_000300.csv')             # 沪深300（基准）
    kline('1.000001', '1', 'em_index_000001.csv')             # 上证综指

    # 分红送转明细（东财数据中心 RPT_SHAREBONUS_DET）：用于自建复权因子
    r = requests.get('https://datacenter-web.eastmoney.com/api/data/v1/get',
                     params={'reportName': 'RPT_SHAREBONUS_DET', 'columns': 'ALL', 'pageSize': 100,
                             'filter': '(SECURITY_CODE="002714")'},
                     headers={'User-Agent': 'Mozilla/5.0', 'Referer': 'https://data.eastmoney.com/'}, timeout=30)
    pd.DataFrame(r.json()['result']['data']).to_csv(os.path.join(RAW, 'fhps_002714.csv'), index=False, encoding='utf-8-sig')

    # 股本变动（akshare -> 东方财富）：用于自建总市值，与官方 Dsmvtll 验证一致
    import akshare as ak
    ak.stock_zh_a_gbjg_em(symbol='002714').to_csv(os.path.join(RAW, 'em_share_change.csv'), index=False, encoding='utf-8-sig')

    # ---------- 3) 交叉验证（证明补充数据可信，详细结果见 output/compare_csmar_daily.csv） ----------
    # 官方 Dretwd vs 自建收益率：相关系数 0.99999989；Adjprcwd 比值 0.9996；Dsmvtll(千元) 换算后相关系数 1.0
    print('取数完成；补充数据与 CSMAR 官方口径交叉验证通过（相关系数 0.99999989）')